# ============================================================
 FIFA WORLD CUP 2026 - BALL POSSESSION ANALYSIS

 Analytic Question:
 Is there a significant difference in the average
 ball possession percentage between winning and
 losing teams in FIFA World Cup 2026 matches?
# ============================================================

In [25]:
# ============================================================
# Import libraries
# ============================================================

import pandas as pd

In [26]:
# ============================================================
# Load dataset
# ============================================================

file_path = "FIFA_WC2026_Possession.csv"
df_src = pd.read_csv(file_path)

# Check rows and columns of dataset
print("Rows:", df_src.shape[0])
print("Columns:", df_src.shape[1])

# View dataset info
print("\n----First 5 rows: ----")
print(df_src.head())
print("\n")
print(df_src.info())

Rows: 208
Columns: 9

----First 5 rows: ----
   MatchID        Date                      Match            Team  \
0        1  2026-06-11     Mexico vs South Africa          Mexico   
1        1  2026-06-11     Mexico vs South Africa    South Africa   
2        2  2026-06-11  Korea Republic vs Czechia  Korea Republic   
3        2  2026-06-11  Korea Republic vs Czechia         Czechia   
4        3  2026-06-12      Canada vs Bosnia–Herz          Canada   

         Opponent  Goals_For  Goals_Against Result  Ball_Possession_pct  
0    South Africa          2              0    Win                   61  
1          Mexico          0              2   Loss                   40  
2         Czechia          2              1    Win                   62  
3  Korea Republic          1              2   Loss                   38  
4     Bosnia–Herz          1              1   Draw                   61  


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 

In [27]:
# ============================================================
# Clean whitespace, check missing values, and check duplicated rows
# ============================================================

# Remove leading and trailing whitespace
df_src["Team"] = df_src["Team"].str.strip()
df_src["Opponent"] = df_src["Opponent"].str.strip()

print("Number of unique teams:", df_src["Team"].nunique())

# Check missing values
print("\n----Check missing values: ----")
print(df_src.isnull().sum())

# Check duplicated rows
dup = df_src.duplicated().sum()
print("\nNumber of duplicated rows:", dup)

Number of unique teams: 48

----Check missing values: ----
MatchID                0
Date                   0
Match                  0
Team                   0
Opponent               0
Goals_For              0
Goals_Against          0
Result                 0
Ball_Possession_pct    0
dtype: int64

Number of duplicated rows: 0


In [28]:
# ============================================================
# Validate the result using the goals
# ============================================================

# Create expected result based on goals
def get_result(row):
    if row["Goals_For"] > row["Goals_Against"]:
        return "Win"
    elif row["Goals_For"] < row["Goals_Against"]:
        return "Loss"
    else:
        return "Draw"

df_src["Expected_Result"] = df_src.apply(get_result, axis=1)

# Compare expected result with recorded result
result_err = df_src[
    df_src["Result"] != df_src["Expected_Result"]
]

print("Number of inconsistent results:", len(result_err))

if len(result_err) > 0:
    print(result_err[
        ["MatchID", "Team", "Goals_For", "Goals_Against",
         "Result", "Expected_Result"]
    ])

Number of inconsistent results: 0


In [29]:
# Remove temporary column
df_src.drop(columns=["Expected_Result"], inplace=True)

In [30]:
# ============================================================
# Select the necessary columns for the analysis
# ============================================================

df_main = df_src[["MatchID", "Match", "Team", "Result", "Ball_Possession_pct"]]

print("----First 5 rows after removing unnecessay columns: ----")
print(df_main.head())

print("\nRows:", df_main.shape[0])
print("Columns:", df_main.shape[1])

----First 5 rows after removing unnecessay columns: ----
   MatchID                      Match            Team Result  \
0        1     Mexico vs South Africa          Mexico    Win   
1        1     Mexico vs South Africa    South Africa   Loss   
2        2  Korea Republic vs Czechia  Korea Republic    Win   
3        2  Korea Republic vs Czechia         Czechia   Loss   
4        3      Canada vs Bosnia–Herz          Canada   Draw   

   Ball_Possession_pct  
0                   61  
1                   40  
2                   62  
3                   38  
4                   61  

Rows: 208
Columns: 5


In [31]:
# ============================================================
# Check result values in each categories
# ============================================================

print("----Match Result Category Counts: ----")
print(df_main["Result"].value_counts())

----Match Result Category Counts: ----
Result
Win     80
Loss    80
Draw    48
Name: count, dtype: int64


In [32]:
# ============================================================
# Check ball possession range should be between 0-100%
# ============================================================

invalid_poss = df_main[
    (df_main["Ball_Possession_pct"] < 0) |
    (df_main["Ball_Possession_pct"] > 100)
]

print("----Invalid possession values: ----")
print(invalid_poss)

----Invalid possession values: ----
Empty DataFrame
Columns: [MatchID, Match, Team, Result, Ball_Possession_pct]
Index: []


In [33]:
# ============================================================
# Ball Possession Validation
# ============================================================

# Check the total ball possession percentage for each match
poss_check = df_main.groupby(["MatchID", "Match"])["Ball_Possession_pct"].sum()

print(poss_check.describe())

# Filter and show only the matches with total Ball Possession not totaling 100%
print("\n----Matches with Total Ball Possession not totaling 100%: ----\n")

invalid_total = poss_check[poss_check != 100]

print(invalid_total)
print("\nNumber of matches not totaling 100%:", len(invalid_total))

count    104.00000
mean     100.12500
std        0.33232
min      100.00000
25%      100.00000
50%      100.00000
75%      100.00000
max      101.00000
Name: Ball_Possession_pct, dtype: float64

----Matches with Total Ball Possession not totaling 100%: ----

MatchID  Match                         
1        Mexico vs South Africa            101
24       Uzbekistan vs Colombia            101
31       Türkiye vs Paraguay               101
42       France vs Iraq                    101
48       Colombia vs Congo DR              101
50       Bosnia–Herz vs Qatar              101
53       South Africa vs Korea Republic    101
57       Tunisia vs Netherlands            101
59       Türkiye vs USA                    101
83       Spain vs Austria                  101
84       Portugal vs Croatia               101
87       Argentina vs Cabo Verde           101
93       Portugal vs Spain                 101
Name: Ball_Possession_pct, dtype: int64

Number of matches not totaling 100%: 13


After verifying the ball possession data for each match against the original website, all values were found to be correct. Therefore, no changes were made to the Ball_Possession_pct column.

In [34]:
# ============================================================
# Check the main variable stats
# ============================================================

print(df_main["Ball_Possession_pct"].describe())

count    208.000000
mean      50.062500
std       13.278181
min       21.000000
25%       39.750000
50%       50.000000
75%       61.000000
max       79.000000
Name: Ball_Possession_pct, dtype: float64


In [35]:
# ============================================================
# Save cleaned dataset as a new CSV file
# ============================================================

df_main.to_csv(
    "FIFA_WC2026_Possession_Cleaned.csv",
    index=False
)

print("CSV file saved successfully.")

CSV file saved successfully.
